**Install PANNA package and requirements:**


In [10]:
import os, sys, subprocess
from pathlib import Path
from ase.io import read
from ase.visualize import view
from scripts import *

def find_project_root(target="CECAM_LATTE"):
    cwd = Path.cwd()

    for parent in [cwd] + list(cwd.parents):
        if parent.name == target:
            return parent
            
    for p in cwd.rglob(target):
        if p.is_dir():
            return p

    raise RuntimeError(f"{target} not found anywhere")

main_dir = find_project_root()
print(main_dir)

/home/max/CECAM_LATTE


In [15]:
!{sys.executable} -m pip install {main_dir}/PANNA

Processing /home/max/CECAM_LATTE/PANNA
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for PANNA: filename=panna-2.1.0-py3-none-any.whl size=140731 sha256=a9ad918708aaec29782f4061cd1b664d30cdec1ac10c7d3a1f9b5564457f83fb
  Stored in directory: /tmp/pip-ephem-wheel-cache-oatw2sr4/wheels/8c/37/9e/d40647326845463ec9c0b02b0b75a2e52806f96605208efd14
Successfully built PANNA
  Attempting uninstall: PANNA
    Found existing installation: PANNA 2.1.0
    Uninstalling PANNA-2.1.0:
      Successfully uninstalled PANNA-2.1.0


**Let's visualize the Carbon dataset:**

In [5]:
file = os.path.join(main_dir, 'dia_100/datasets/training_100_dia.xyz')
trajectory = read(file, index=":")
view(trajectory)

<Popen: returncode: None args: ['/home/max/.conda/envs/metatomic/bin/python'...>

**Step 1: train LATTE on a small diamond dataset.**

training set: 80 atomic structures, test set: 20 structures.

In [6]:
os.chdir(f'{main_dir}/dia_100/training')
!cat train.ini

[IO_INFORMATION]
# training dir, models are here under /models folder
train_dir = .  
# path to the training set
data_dir = ../datasets/training_set
# path to validation set
val_data_dir = ../datasets/validation_set
# set True if you would like to restart a training 
load_weights = True
# uncomment if you would like to start from a specific epoch
#weights_file = models/epoch_2110_step_2110000.pkl
# how frequently to save the model
save_epoch_freq = 50
# for accelerating the training
data_cache = True
[DATA_INFORMATION]
# list of elements present in training set
atomic_sequence = C
input_format = example   
output_offset = -253.67376683573406 

[MODEL_INFORMATION]
## Our standard model
model_type = MLP
## Architecture: l1:l2:l3
architecture = 32:16:1
# which descriptor, PANNA or LATTE
descriptor_type = LATTE
# more technical details, leave them as they are
nn_mode = list
pair_mod = 500
at_mod = 50
# the radius of atomic interaction
Rc = 5.0
sig = 1.0
# number of n-body interactions
desc

In [7]:
!{sys.executable} -m panna.train_jax -c train.ini

I0000 00:00:1782834180.132878    7616 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782834181.189647    7616 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782834186.468322    7616 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1782834188.929264    7616 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
INFO - 
    ____   _    _   _ _   _    _           
   |  _ \ / \  | \ | | \ | |  / \     
   | |_) / _ \ |  \| |  \| | / _ \     
   |  __/ ___ \| |\  | |\  |/ ___ \    
   |_| /_/   \_\_| \_|_| \_/_/   \_\ 

 Properties from Artificial Neural Network Architectures

INFO - reading train.ini
I0000 00:00:17828

Visualize the training process:

In [16]:
fig, axs = plt.subplots(1, 2, figsize = (12, 5))
plot_training(f'{main_dir}/dia_100/training', axs)
plot_setup(fig, axs, log=True, xlim=2000, 
            ylim=0.1, fylim=0.3,
            title='Trained on 100 Diamond')

ImportError: cannot import name 'RenderMode' from 'matplotlib.ft2font' (/home/max/.conda/envs/metatomic/lib/python3.13/site-packages/matplotlib/ft2font.cpython-313-x86_64-linux-gnu.so)

Now, let's make prediction on Diamond test set.
Is this model able to predict more diverse dataset?

In [18]:
os.chdir(f'{main_dir}/dia_100/test_dia')
!{sys.executable} -m panna.train_jax -c train.ini
os.chdir(f'{main_dir}/dia_100/test_div')
!{sys.executable} -m panna.train_jax -c train.ini

I0000 00:00:1782835761.564410    9524 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782835762.161258    9524 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782835767.605165    9524 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1782835770.470951    9524 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
INFO - 
    ____   _    _   _ _   _    _           
   |  _ \ / \  | \ | | \ | |  / \     
   | |_) / _ \ |  \| |  \| | / _ \     
   |  __/ ___ \| |\  | |\  |/ ___ \    
   |_| /_/   \_\_| \_|_| \_/_/   \_\ 

 Properties from Artificial Neural Network Architectures

INFO - reading train.ini
Looking models in

Test the performance of the model on familiar & unfamiliar structures:

In [20]:
path = f'{main_dir}/dia_100/test_dia'
epoch = 50
Force_mae, Energy_mae= Force_MAE(path, epoch), Energy_MAE(path, epoch) 
print(f'At epoch {epoch}: \n Force_MAE: {Force_mae:.3f} \n Energy_MAE: {Energy_mae:.3f}')

At epoch 50: 
 Force_MAE: 0.120 
 Energy_MAE: 0.021


In [21]:
path = f'{main_dir}/dia_100/test_div'
epoch = 50
Force_mae, Energy_mae= Force_MAE(path, epoch), Energy_MAE(path, epoch) 
print(f'At epoch {epoch}: \n Force_MAE: {Force_mae:.3f} \n Energy_MAE: {Energy_mae:.3f}')

At epoch 50: 
 Force_MAE: 0.839 
 Energy_MAE: 0.344
